# BRFSS 2015 — Colab research runner
Default: **NEW_PROTOCOL**, a new internal-validation study, not a restoration or independent validation of old results. **LEGACY_RESUME** fails unless the complete immutable legacy bundle passes its audit. Synthetic mode validates software only. Final-test selection is forbidden. Source-only GitHub downloads support NEW_PROTOCOL.

Set optional environment variables `BRFSS_REPO`, `BRFSS_DATA`, `BRFSS_CODEBOOK`, `BRFSS_PROTOCOL`, `BRFSS_PROFILE`, `BRFSS_OUT` before Run all. Python 3.10–3.12 is supported; other versions require dependency revalidation. If setup changes imported binary packages, restart the session and Run all again.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

SYNTHETIC = os.environ.get("BRFSS_SYNTHETIC", "0") == "1"
if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError(f"Python {sys.version.split()[0]} is outside tested 3.10–3.12. Select a compatible Colab runtime; dependency validation is required for newer Python.")
if not SYNTHETIC:
    from google.colab import drive
    drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
explicit = os.environ.get("BRFSS_REPO")
candidates = [Path(explicit)] if explicit else [
    Path.cwd(), Path.cwd().parent,
    drive_root / "PPTK-main (1)/PPTK-main/BRFSS2015-Statistical-Methods",
    drive_root / "BRFSS2015-Statistical-Methods",
    drive_root / "BRFSS2015_Statistical_Methods_AZ/BRFSS2015-Statistical-Methods",
]
if not explicit and drive_root.is_dir():
    candidates += sorted(drive_root.glob("PPTK-main*/PPTK-main/BRFSS2015-Statistical-Methods"))
REPO = next((p.resolve() for p in candidates if (p / "pyproject.toml").is_file()), None)
if REPO is None:
    raise FileNotFoundError(f"Set BRFSS_REPO to the project folder. Checked: {candidates}")
required = ["src/data/ingest.py", "src/data/outcomes.py", "src/data/synthetic.py", "scripts/colab_runtime.py", "configs/project.yaml"]
missing = [name for name in required if not (REPO / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete source checkout: {missing}. Update the repository; no missing files will be synthesized.")
os.chdir(REPO)
sys.path.insert(0, str(REPO)) if str(REPO) not in sys.path else None
print("Repository:", REPO, "Python:", sys.version.split()[0])

In [ ]:
if not SYNTHETIC:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    if os.environ.get("BRFSS_INSTALL_OPTIONAL", "1") == "1":
        for requirement in ["requirements-boosters.txt", "requirements-research-extensions.txt"]:
            subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirement], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
from scripts.verify_environment import verify
versions = verify()
# Import every project module before accessing data.
import importlib
import pkgutil
for package_name in ["src", "src_extensions"]:
    package = importlib.import_module(package_name)
    for module in pkgutil.walk_packages(package.__path__, package_name + "."):
        importlib.import_module(module.name)

In [ ]:
from scripts.colab_runtime import execute, locate_data
PROTOCOL = os.environ.get("BRFSS_PROTOCOL", "NEW_PROTOCOL")
PROFILE = os.environ.get("BRFSS_PROFILE", "quick").lower()
MODE = os.environ.get("BRFSS_MODE", "binary")
SEED = int(os.environ.get("BRFSS_SEED", "20261107"))
OUT = Path(os.environ.get("BRFSS_OUT", str(REPO / "runs" / ("new_protocol_synthetic" if SYNTHETIC else "new_protocol_2015"))))
DATA = None if SYNTHETIC else str(locate_data(os.environ.get("BRFSS_DATA")))
CODEBOOK = os.environ.get("BRFSS_CODEBOOK")
if not SYNTHETIC and not CODEBOOK:
    from src.data.ingest import discover
    CODEBOOK = str(discover(directories=[Path(DATA).parent], filename="codebook15_llcp.pdf"))
execute(protocol=PROTOCOL, synthetic=SYNTHETIC, data=DATA, codebook=CODEBOOK,
        out=OUT, profile=PROFILE, mode=MODE, seed=SEED)

In [ ]:
from src.utils.io import read_json
from IPython.display import display, Markdown
if PROTOCOL == "NEW_PROTOCOL":
    ledger = read_json(OUT / "test_access.json")
    assert ledger["state"] == "COMPLETE" and ledger["evaluation_count"] == 1
    display(Markdown((OUT / "final_report.md").read_text()))
    print(read_json(OUT / "protocol.json"))
else:
    print(read_json(OUT / "integrity.json"))
print("Outputs:", OUT)
print("SYNTHETIC SOFTWARE VALIDATION ONLY" if SYNTHETIC else "Real-data run; review audit and protocol scope before interpreting results.")
print("COLAB_RUNTIME_NOT_VERIFIED by repository CI; an executed notebook in your actual Colab session is separate evidence.")